In [0]:
# Preâmbulo para configurar o catálogo
dbutils.widgets.text("catalog", "v_credit_case")
catalog = dbutils.widgets.get("catalog")

bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Catálogo configurado com sucesso!")

In [0]:
# Imports usados nas funções da camada
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
# Função para carregar um DataFrame a partir do nome de uma tabela da silver
def load_df(table_name):
    # Se der erro, a execução deve parar
    try:
        existing = {t.name for t in spark.catalog.listTables(silver_schema)}
    except Exception:
        raise FileNotFoundError(f"O schema {silver_schema} não existe ou não está acessível.")
    if table_name not in existing:
        raise FileNotFoundError(
            f"{table_name} não foi encontrado em {silver_schema}. "
            f"Tabelas disponíveis: {sorted(existing)}"
        )

    return spark.table(f"{silver_schema}.{table_name}")

In [0]:
# Função para salvar o DataFrame na camada gold
def save_df(df, table_name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
            .saveAsTable(f"{gold_schema}.{table_name}")

In [0]:
# Recebe o DataFrame e devolve com uma chave sequencial gerada
def add_sequential_key(df, key_name, order_by=None):
    """Acrescenta a chave sequencial (1..N) como primeira coluna do DataFrame.

    order_by define a ordem que numera as linhas; por padrão usa todas as colunas,
    o que mantém a numeração reprodutível entre execuções.
    """
    if key_name in df.columns:
        raise ValueError(f"A coluna {key_name} já existe no DataFrame.")

    sort_columns = order_by or df.columns
    window_spec = Window.orderBy(*[F.col(c) for c in sort_columns])
    keyed = df.withColumn(key_name, F.row_number().over(window_spec))

    return keyed.select(F.col(key_name), *[F.col(c) for c in df.columns])